<a href="https://colab.research.google.com/github/Daniel-534/MecanicaCeleste/blob/main/Clases/Clase20260430-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cuaderno de clase
## Mecánica Celeste (2026-1) con Jorge I. Zuluaga
## El problema de los dos cuerpos en el tiempo

In [ ]:
!pip install -Uq pymcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.6 MB/s eta 0:00:00


In [ ]:
import pymcel as pc
import numpy as np
deg = np.pi / 180

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


In [ ]:
e	= 0.1911663355386932
a	= 0.9223803173917017 * pc.constantes.au
q	= 0.7460522521429133 * pc.constantes.au
i	= 3.340958441017069*deg
node = 203.8996515621043*deg
peri = 126.6728325163065*deg
tp = 2461042.918242006079 * 86400 # fecha juliana
mu = pc.constantes.mu_sun # Este es el mu de todo el sistema solar

Varaibles derivadas:

In [ ]:
h = np.sqrt(mu * a * (1 - e**2))
b = a * np.sqrt(1 - e**2)

h, b

(np.float64(4200387442699322.5), np.float64(135441343761.16716))

¿Cuál es el tiempo?

In [ ]:
from astropy.time import Time

In [ ]:
t = Time("2029-04-13 18:52:00", scale='tdb').jd * 86400
t

np.float64(212737560720.0)

Escribir la ecuación en la forma: f(E) = 0

In [ ]:
def ecuacion_kepler(E):
  funcion = E - e*np.sin(E) - h/(a*b)*(t-tp)
  return funcion

Resolver la ecuación de Kepler:

In [ ]:
from scipy.optimize import newton

In [ ]:
E = newton(ecuacion_kepler, 0)
E

np.float64(23.081594014937842)

Obtenemos ahora la anomalía verdadera:

In [ ]:
f = 2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(E/2))
f

np.float64(-2.214590394976548)

Una vez tenemos la anomalía verdadera podemos calcular la posición en su sistema perifocal:

In [ ]:
p = a*(1-e**2)
r = p / (1 + e * np.cos(f))
xf = r * np.cos(f)
yf = r * np.sin(f)
zf = 0

Usando la rotación en el espacio:

In [ ]:
import spiceypy as spy
R = spy.eul2m(-node, -i, -peri, 3, 1, 3)

r = R @ np.array([xf, yf, zf])
r

array([-1.37524466e+11, -6.03294163e+10, -3.26653523e+07])

Vamos a comparar la posición con la que nos da consulta horizons:


In [ ]:
tabla, jd, X = pc.consulta_horizons(id='Apophis', location='@SSB', epochs='2029-04-13 18:52:00')
X[:3]

/usr/local/lib/python3.12/dist-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


array([-1.37263651e+11, -6.04834602e+10, -4.52634041e+06])